# Lab 7A — A Supervisor Delegating to Two Workers

Every agent so far has been one loop with several tools. This one splits into a
**supervisor** that owns no tools at all and two **workers** that each own one
capability.

| Cell group | What you do |
|---|---|
| 1–2 | Build two workers: one with Genie, one with the policy index. |
| 3 | Build a supervisor whose only tool is `delegate`. |
| 4 | Ask something that needs both — and watch it **re-delegate**. |
| 5 | **Starve a worker of context and see how it fails.** |
| 6 | Read the hand-offs from the span tree. |
| 7 | Weigh the pattern honestly. |

**Compute:** Serverless. **Prerequisites:** [Lab 3A](#) (index), [Lab 4A](#) (Genie).

> ⚠️ **Each supervisor run is 60–90 seconds.** Genie is slow, and every delegation
> adds a full model round-trip on top of the worker's own call.

In [0]:
%pip install -q "mlflow[databricks]" databricks-ai-search openai
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 1. Setup — and the two capabilities

The analytics worker gets Genie and **no** policy access. The policy worker gets the
vector index and **no** data access. Neither can do the other's job, which is what
makes the supervisor necessary rather than decorative.

In [0]:
import json, os, time, warnings, requests
import mlflow
from databricks.sdk import WorkspaceClient
from mlflow.entities.trace_location import UnityCatalog
from openai import OpenAI
from databricks.ai_search.client import VectorSearchClient

warnings.filterwarnings("ignore", message=".*serializer warnings.*")
CATALOG, SCHEMA = "agents_labs", "retail"
MODEL = "databricks-claude-sonnet-5"
SPACE_TITLE = "Retail Customer Orders"

w = WorkspaceClient()
user = w.current_user.me().user_name
HOST = w.config.host
TOKEN = w.config.authenticate()["Authorization"].removeprefix("Bearer ")
HEAD = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
client = OpenAI(api_key=TOKEN, base_url=f"{HOST}/serving-endpoints")
_ix = VectorSearchClient(workspace_url=HOST, personal_access_token=TOKEN,
                         disable_notice=True).get_index(
    endpoint_name="agents-labs-vs", index_name=f"{CATALOG}.{SCHEMA}.support_chunks_idx")

whs = [x for x in w.warehouses.list()]
os.environ["MLFLOW_TRACING_SQL_WAREHOUSE_ID"] = whs[0].id
EXPERIMENT = f"/Users/{user}/agents-labs-7a"
mlflow.set_experiment(experiment_name=EXPERIMENT,
                      trace_location=UnityCatalog(catalog_name=CATALOG, schema_name=SCHEMA))


def api(method, path, **kw):
    r = requests.request(method, f"{HOST}{path}", headers=HEAD, timeout=120, **kw)
    r.raise_for_status()
    return r.json() if r.text else {}


spaces = api("GET", "/api/2.0/genie/spaces").get("spaces", [])
SPACE_ID = next(s["space_id"] for s in spaces if s.get("title") == SPACE_TITLE)
print(f"  genie space : {SPACE_ID}")
print(f"  experiment  : {EXPERIMENT}")

  genie space : 01f1bc6b0a4f1c6d8269cc9c1ec2af08
  experiment  : /Users/admin@datacouchoutlook.onmicrosoft.com/agents-labs-7a


## 2. The two workers

Each is decorated `@mlflow.trace(span_type="AGENT")`. That decorator is what makes
the hand-offs visible in cell 6 — without it, a multi-agent run is a black box.

In [0]:
TERMINAL = {"COMPLETED", "FAILED", "QUERY_RESULT_EXPIRED", "CANCELLED"}


def _ask_genie(question, timeout=180):
    base = f"/api/2.0/genie/spaces/{SPACE_ID}"
    conv = api("POST", f"{base}/start-conversation", json={"content": question})
    cid, mid = conv["conversation_id"], conv["message_id"]
    path = f"{base}/conversations/{cid}/messages/{mid}"
    for _ in range(timeout):
        m = api("GET", path)
        if m.get("status") in TERMINAL:
            break
        time.sleep(1)
    out = {"text": None, "sql": None, "rows": None}
    for att in (m.get("attachments") or []):
        if att.get("text", {}).get("content"):
            out["text"] = att["text"]["content"]
        if att.get("query"):
            out["sql"] = att["query"].get("query")
            try:
                res = api("GET", f"{path}/attachments/{att['attachment_id']}/query-result")
                sr = res.get("statement_response", {})
                cols = [c["name"] for c in sr.get("manifest", {}).get("schema", {}).get("columns", [])]
                out["rows"] = [dict(zip(cols, r))
                               for r in (sr.get("result", {}).get("data_array") or [])]
            except Exception as e:
                out["rows"] = [{"error": str(e)[:120]}]
    return out


@mlflow.trace(span_type="AGENT")
def analytics_worker(task: str) -> dict:
    """Owns the numbers. Has Genie. No policy access."""
    g = _ask_genie(task)
    return {"worker": "analytics", "task": task,
            "finding": g["text"], "sql": g["sql"], "rows": g["rows"]}


def _text(m):
    c = m.content
    if c is None: return ""
    if isinstance(c, str): return c
    return "\n".join(b.get("text", "") for b in c
                     if isinstance(b, dict) and b.get("type") == "text").strip()


@mlflow.trace(span_type="AGENT")
def policy_worker(task: str) -> dict:
    """Owns the written rules. Has the index. No data access."""
    r = _ix.similarity_search(query_text=task, columns=["doc_id", "title", "chunk"],
                              filters={"audience": "customer"}, num_results=3)
    hits = [{"doc_id": x[0], "title": x[1], "text": x[2]}
            for x in (r.get("result", {}).get("data_array", []) or [])]
    ctx = "\n\n".join(f"[{h['doc_id']}] {h['text']}" for h in hits)
    msg = client.chat.completions.create(
        model=MODEL, max_tokens=600,
        messages=[{"role": "system", "content":
                   "You summarise company policy. Answer only from the excerpts. "
                   "Cite document ids. If they do not cover it, say so."},
                  {"role": "user", "content": f"Excerpts:\n{ctx}\n\nTask: {task}"}],
    ).choices[0].message
    return {"worker": "policy", "task": task, "finding": _text(msg),
            "cited": [h["doc_id"] for h in hits]}


WORKERS = {"analytics": analytics_worker, "policy": policy_worker}
print("  workers:", list(WORKERS))

  workers: ['analytics', 'policy']


## 3. The supervisor

It has exactly one tool. It cannot query data or read policy itself — that
constraint is the design. It forces every fact in the final answer to come from a
worker that is accountable for it.

**The tool description is the routing logic.** There is no router model and no
classifier; an imprecise description produces mis-routing, and the fix is editing
prose.

In [0]:
TOOLS = [{"type": "function", "function": {
    "name": "delegate",
    "description": "Delegate one self-contained sub-task to a specialist worker. "
                   "'analytics' can query governed order and customer data. "
                   "'policy' can search written company policy. Neither can do the "
                   "other's job.",
    "parameters": {"type": "object", "properties": {
        "worker": {"type": "string", "enum": ["analytics", "policy"]},
        "task": {"type": "string",
                 "description": "A complete, standalone instruction. The worker cannot "
                                "see the conversation or other workers' findings."}},
        "required": ["worker", "task"]}}}]

SUPERVISOR = """You are a supervisor coordinating two specialist workers.

You cannot query data or read policy yourself. You must delegate.

Rules:
  * Each delegated task must stand alone. A worker sees only the task string,
    never the conversation and never another worker's findings.
  * If a task depends on something you do not know yet, delegate the first
    question, read the answer, then delegate the second with that fact included.
  * Finish with one combined answer that says which worker produced which part."""


@mlflow.trace(span_type="AGENT")
def run(request: str, max_steps: int = 6) -> str:
    messages = [{"role": "system", "content": SUPERVISOR},
                {"role": "user", "content": request}]
    print(f"\n  request: {request}\n")
    for step in range(1, max_steps + 1):
        msg = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS, max_tokens=1100
        ).choices[0].message
        messages.append(msg.model_dump(exclude_none=True))
        if not msg.tool_calls:
            print(f"  step {step}: supervisor combines the findings")
            return _text(msg)
        for tc in msg.tool_calls:
            a = json.loads(tc.function.arguments or "{}")
            wk, task = a.get("worker"), a.get("task", "")
            print(f"  step {step}: delegate -> {wk}")
            print(f"           task: {task[:92]}")
            out = WORKERS[wk](task) if wk in WORKERS else {"error": f"no worker {wk}"}
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(out, default=str)[:9000]})
    return "(no answer within max_steps)"


print("  supervisor ready")

  supervisor ready


## 4. A question that needs both, in order

> *"Order ORD-1007 — look up what was ordered and by whom, then tell me whether that
> customer would pay a return shipping fee if they sent it back."*

**What to watch:** the number of delegations, and whether any worker is asked twice.

In [0]:
print(run("Order ORD-1007 — look up what was ordered and by whom, then tell me whether "
          "that customer would pay a return shipping fee if they sent it back."))


  request: Order ORD-1007 — look up what was ordered and by whom, then tell me whether that customer would pay a return shipping fee if they sent it back.

  step 1: delegate -> analytics
           task: Look up order ORD-1007. Provide: the items ordered, the customer name/ID who placed it, the 
  step 1: delegate -> policy
           task: What is the company policy on return shipping fees? Specifically, under what circumstances d
[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
  step 2: delegate -> analytics
           task: For order ORD-1007 (customer CUST-006, item: task chair), what is the quantity of task chair
  step 3: supervisor combines the findings
## Answer

**Order details (from Analytics):**
- Order **ORD-1007** was placed on 2026-08-11 by customer **Manchester Fitout** (CUST-006).
- Item ordered: **task

Trace(trace_id=trace:/agents_labs.retail.1670926665559537/8b23b2c98917a319c7e97311c985252d)

### If it delegated three times, read the third one

In the verified run the sequence was:

```
  step 1: delegate -> analytics    who placed ORD-1007?
  step 2: delegate -> policy       return fees for a plus-tier UK customer?
  step 3: delegate -> analytics    how many units on ORD-1007?
```

**It went back to analytics.** It did not plan three steps up front. The policy
worker's answer mentioned a fit-out exception — 20+ units of one seating product get
free collection regardless of tier — and the supervisor realised it did not know the
quantity. So it asked again, and the answer inverted from *"£12 fee"* to
*"free collection"*.

> ⚠️ **This is emergent, not guaranteed.** Re-run it and you may get two delegations
> and the £12 answer. Nothing in the code forces the re-delegation. If a correct
> answer matters here, you encode the dependency — check quantity *before* quoting a
> fee — rather than hoping the supervisor rediscovers it each time.

## 5. Starve a worker of context

The supervisor prompt insists every task stands alone. Find out what that rule is
protecting you from.

Same worker, same tools. The only difference is whether the task carries its own
context.

In [0]:
import textwrap

BAD  = "Would that customer pay a return shipping fee?"
GOOD = ("A 'plus' tier customer is returning 25 units of a single seating product. "
        "Would they pay a return shipping fee?")

for label, task in (("CONTEXT-DEPENDENT", BAD), ("STANDALONE", GOOD)):
    print(f"\n{'='*74}\n  {label}\n  task: {task}\n{'-'*74}")
    out = policy_worker(task)
    print(textwrap.fill(out["finding"], 72, initial_indent="  ",
                        subsequent_indent="  ")[:1100])
    print(f"\n  cited: {out['cited']}")


  CONTEXT-DEPENDENT
  task: Would that customer pay a return shipping fee?
--------------------------------------------------------------------------
[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
  Based on the excerpts, whether a return shipping fee applies depends
  on the customer's tier (DOC-001):  - **Premier tier customers**:
  Return shipping is **free**. - **Standard and Plus tier customers**: A
  **flat 12 GBP collection fee** is charged, which is deducted from the
  refund.  **Exceptions:** - If the item is marked as **final sale**, it
  cannot be returned at all, so the fee question would not apply. - If
  the item **arrived damaged**, a replacement or refund is offered
  regardless of the 30-day window, though the excerpt does not specify
  whether the shipping fee is waived in this case — this detail is n

[Trace(trace_id=trace:/agents_labs.retail.1670926665559537/1a37fd5f121e8cf1ca01dec87edb6991), Trace(trace_id=trace:/agents_labs.retail.1670926665559537/648564eb31e32efe15b73db28d74bd63)]

### Compare the `cited` lines, not the prose

The vague task did **not** refuse. It did not say *"I don't know who you mean."* It
produced a fluent, correctly-cited, tier-accurate answer.

It also never retrieved **DOC-007**, the fit-out exception — because the task string
*is* the retrieval query, and *"would that customer pay a fee"* contains no signal
for *25 units* or *seating*.

> 🚨 **This is the failure mode that survives review.** A starved worker answers the
> general question instead of the specific one, cites real documents for it, and
> hands back something a reviewer will accept. In a multi-agent system the supervisor
> *is* that reviewer, and it has no way to know a document it never saw exists.
>
> **When a worker's task doubles as a retrieval query, the supervisor's phrasing is a
> retrieval-quality problem, not a prose problem.**

## 6. Read the hand-offs from the span tree

Printing from inside the supervisor works for one run on your screen. It does not
work for a run that happened yesterday in a job. The traces do.

In [0]:
mlflow.flush_trace_async_logging()
exp = mlflow.get_experiment_by_name(EXPERIMENT)
df = mlflow.search_traces(locations=[exp.experiment_id], max_results=5)
print(f"  trace_location : {exp.trace_location}")
print(f"  traces         : {len(df)}\n")

tr = mlflow.get_trace(df.trace_id.iloc[0])
by_id = {s.span_id: s for s in tr.data.spans}


def depth(s):
    d = 0
    while s.parent_id and s.parent_id in by_id:
        s = by_id[s.parent_id]; d += 1
    return d


for s in tr.data.spans:
    ins = s.inputs
    if isinstance(ins, str):
        try: ins = json.loads(ins)
        except Exception: ins = {"task": ins}
    extra = ""
    if s.name in ("analytics_worker", "policy_worker"):
        extra = f"   task: {str((ins or {}).get('task',''))[:56]}"
    dur = (s.end_time_ns - s.start_time_ns) / 1e9
    print(f"  {'  '*depth(s)}{s.name:18} [{s.span_type}] {dur:6.2f}s{extra}")

  trace_location : UnityCatalog(catalog_name='agents_labs', schema_name='retail', table_prefix='1670926665559537')
  traces         : 5

  policy_worker      [AGENT]   6.28s   task: Would that customer pay a return shipping fee?


## 7. Weigh the pattern honestly

| | Single agent, both tools | Supervisor + workers |
|---|---|---|
| Latency | one loop, ~25s | **~60s**, 3 hops |
| Model calls | ~3 | ~6 |
| Accountability | one context holds everything | each fact traces to a named worker |
| Governance | one identity needs every grant | each worker can be its own principal |
| Prompt size | grows with every tool | each worker's prompt stays small |
| Failure mode | tool confusion | **silent context starvation** (cell 5) |

> 💡 **The pattern earns its cost when workers need different privileges.** An
> analytics worker touching governed tables under a principal that has no business
> reading internal policy is a real separation, enforced by Unity Catalog rather than
> by a prompt — [Lab 5B](#) showed each worker can be a distinct service principal
> whose MCP tool list is filtered by its grants.
>
> It is **not** worth 2.5× the latency purely to tidy up your prompts.

### Try this before you move on

1. Re-run cell 4 three times. Count the delegations each time.
2. Change the `delegate` description to something vague and re-run. Does it still
   route correctly?
3. Ask the analytics worker a policy question directly. It has no policy access —
   what does it do?